In [9]:
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display

In [10]:
df = pd.read_csv(
    "/workspaces/pcf-quality-framework/BEACON-PCF/data/PublicTablesForCarbonCatalogueDataDescriptor_v30Oct2021(Product Level Data).csv",
    encoding="latin1"
)

print(f"Dataset shape: {df.shape}")

Dataset shape: (866, 25)


In [11]:
# Helper Functions


def clean_string_series(series):
    return series.astype("string").str.strip()


def is_missing_or_blank(series):
    return series.isna() | clean_string_series(series).eq("")


def calculate_score(passed, checked):
    if checked == 0:
        return np.nan
    return round((passed / checked) * 100, 2)


def percentage_to_float(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text.endswith("%"):
        try:
            return float(text[:-1].strip())
        except ValueError:
            return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    return np.nan


def text_quality_mask(series):
    return (
        (~series.isna())
        & clean_string_series(series).ne("")
        & series.map(lambda x: isinstance(x, str))
    )

In [12]:
# R01 — Missing Value Check | Completeness


def rule_01_missing_values(df):
    results = []
    total_rows = len(df)

    for column in df.columns:

        # Optional fields are excluded from completeness scoring
        if column in OPTIONAL_COLUMNS:
            continue

        missing = int(is_missing_or_blank(df[column]).sum())
        present = total_rows - missing

        results.append({
            "Feature": column,
            "Missing/Blank Values": missing,
            "Present Values": present,
            "Completeness (%)": calculate_score(present, total_rows),
            "Status": "PASS" if missing == 0 else "WARNING"
        })

    return pd.DataFrame(results)


r01_results = rule_01_missing_values(df)

r01_score = round(
    r01_results["Completeness (%)"].mean(), 2
)

print("R01 — Missing Value Check")
print(f"Completeness Score: {r01_score}%")

display(r01_results)

R01 — Missing Value Check
Completeness Score: 99.95%


,Feature,Missing/Blank Values,Present Values,Completeness (%),Status
0,*PCF-ID,0,866,100.00,PASS
1,Year of reporting,0,866,100.00,PASS
2,*Stage-level CO2e available,0,866,100.00,PASS
3,Product name (and functional unit),0,866,100.00,PASS
4,Product detail,10,856,98.85,WARNING
5,Company,0,866,100.00,PASS
6,Country (where company is incorporated),0,866,100.00,PASS
7,Company's GICS Industry Group,0,866,100.00,PASS
8,Company's GICS Industry,0,866,100.00,PASS
9,*Company's sector,0,866,100.00,PASS


In [13]:
# R02 — Data Type Validation | Validity

EXPECTED_TYPES = {column: "numeric" for column in NUMERIC_COLUMNS}
EXPECTED_TYPES.update({column: "text" for column in TEXT_COLUMNS})


def rule_02_data_type_validation(df):
    results = []

    for column, expected_type in EXPECTED_TYPES.items():

        series = df[column]
        non_missing = ~is_missing_or_blank(series)

        checked = int(non_missing.sum())

        if expected_type == "numeric":
            converted = pd.to_numeric(series, errors="coerce")
            valid_mask = non_missing & converted.notna()

        else:
            valid_mask = text_quality_mask(series)

        valid = int(valid_mask.sum())
        invalid = checked - valid

        results.append({
            "Feature": column,
            "Expected Type": expected_type,
            "Checked Non-Missing Records": checked,
            "Valid Type Records": valid,
            "Invalid Type Records": invalid,
            "Validity (%)": calculate_score(valid, checked),
            "Status": "PASS" if invalid == 0 else "FAIL"
        })

    return pd.DataFrame(results)


r02_results = rule_02_data_type_validation(df)

r02_score = round(
    r02_results["Validity (%)"].dropna().mean(), 2
)

print("R02 — Data Type Validation")
print(f"Validity Score: {r02_score}%")

display(r02_results)

R02 — Data Type Validation
Validity Score: 100.0%


,Feature,Expected Type,Checked Non-Missing Records,Valid Type Records,Invalid Type Records,Validity (%),Status
0,Year of reporting,numeric,866,866,0,100.0,PASS
1,Product weight (kg),numeric,866,866,0,100.0,PASS
2,"Product's carbon footprint (PCF, kg CO2e)",numeric,866,866,0,100.0,PASS
3,*Carbon intensity,numeric,866,866,0,100.0,PASS
4,*PCF-ID,text,866,866,0,100.0,PASS
5,*Stage-level CO2e available,text,866,866,0,100.0,PASS
6,Product name (and functional unit),text,866,866,0,100.0,PASS
7,Product detail,text,856,856,0,100.0,PASS
8,Company,text,866,866,0,100.0,PASS
9,Country (where company is incorporated),text,866,866,0,100.0,PASS


In [14]:
# R03 — Domain Validation | Validity


DOMAIN_RULES = {
    "*Stage-level CO2e available": [
        "Yes",
        "No"
    ],

    "*Source for product weight": [
        "Estimated from external data based on product description",
        "As per company's submission to CDP"
    ],

    "*Change reason category": [
        "N/a (no %change reported)",
        "N/a (no previous data available)",
        "Product carbon efficiency changed",
        "Model/parameters AND product carbon efficiency changed",
        "Model and/or parameters changed (but not product)"
    ]
}


def rule_03_domain_validation(df):
    results = []

    for column, allowed_values in DOMAIN_RULES.items():

        values = clean_string_series(df[column])
        assessed = ~is_missing_or_blank(df[column])

        valid_mask = assessed & values.isin(allowed_values)

        checked = int(assessed.sum())
        valid = int(valid_mask.sum())
        invalid = checked - valid

        invalid_values = (
            values.loc[assessed & ~values.isin(allowed_values)]
            .dropna()
            .unique()
            .tolist()
        )

        results.append({
            "Feature": column,
            "Allowed Values": "; ".join(allowed_values),
            "Checked Non-Missing Records": checked,
            "Valid Domain Records": valid,
            "Invalid Domain Records": invalid,
            "Validity (%)": calculate_score(valid, checked),
            "Status": "PASS" if invalid == 0 else "FAIL",
            "Invalid Values": (
                ", ".join(map(str, invalid_values))
                if invalid_values else "-"
            )
        })

    return pd.DataFrame(results)


r03_results = rule_03_domain_validation(df)

r03_score = round(
    r03_results["Validity (%)"].dropna().mean(), 2
)

print("R03 — Domain Validation")
print(f"Validity Score: {r03_score}%")

display(r03_results)

R03 — Domain Validation
Validity Score: 98.54%


,Feature,Allowed Values,Checked Non-Missing Records,Valid Domain Records,Invalid Domain Records,Validity (%),Status,Invalid Values
0,*Stage-level CO2e available,Yes; No,866,866,0,100.00,PASS,-
1,*Source for product weight,Estimated from external data based on product ...,866,866,0,100.00,PASS,-
2,*Change reason category,N/a (no %change reported); N/a (no previous da...,866,828,38,95.61,FAIL,No specific reason reported


In [15]:
# R04 — Controlled Vocabulary Check | Consistency


CONTROLLED_VOCABULARY = {
    "Protocol used for PCF": {
        "Traci 2.1": "TRACI 2.1",
        "GHG protocol": "GHG Protocol",
        "Iso 14067": "ISO 14067"
    },

    "Country (where company is incorporated)": {
        "U.S.A": "USA",
        "United States": "USA",
        "UK": "United Kingdom"
    }
}


def rule_04_controlled_vocabulary(df):

    results = []
    details = []

    for column, mapping in CONTROLLED_VOCABULARY.items():

        values = clean_string_series(df[column])
        assessed = ~is_missing_or_blank(df[column])

        inconsistent_mask = assessed & values.isin(mapping.keys())

        checked = int(assessed.sum())
        inconsistent = int(inconsistent_mask.sum())
        consistent = checked - inconsistent

        observed_inconsistent = set(
            values.loc[inconsistent_mask]
        )

        recommended_mapping = "; ".join(
            f"{value} -> {mapping[value]}"
            for value in mapping
            if value in observed_inconsistent
        )

        results.append({
            "Feature": column,
            "Checked Non-Missing Records": checked,
            "Consistent Records": consistent,
            "Non-Standard Records": inconsistent,
            "Consistency (%)": calculate_score(
                consistent,
                checked
            ),
            "Status": "PASS" if inconsistent == 0 else "WARNING",
            "Recommended Mapping": (
                recommended_mapping
                if recommended_mapping else "-"
            )
        })

        # Keep detailed records only for identified inconsistencies
        if inconsistent > 0:

            detail = pd.DataFrame({
                "Feature": column,
                "Observed Value": values.loc[inconsistent_mask]
            })

            detail["Recommended Value"] = (
                detail["Observed Value"].map(mapping)
            )

            details.append(detail)

    report_df = pd.DataFrame(results)

    if details:
        details_df = pd.concat(
            details,
            ignore_index=True
        )
    else:
        details_df = pd.DataFrame()

    return report_df, details_df


r04_results, r04_details = rule_04_controlled_vocabulary(df)

r04_score = round(
    r04_results["Consistency (%)"].dropna().mean(),
    2
)

print("R04 — Controlled Vocabulary Check")
print(f"Consistency Score: {r04_score}%")

display(r04_results)

if not r04_details.empty:
    print("\nNon-standard values:")
    display(r04_details)

R04 — Controlled Vocabulary Check
Consistency Score: 99.94%


,Feature,Checked Non-Missing Records,Consistent Records,Non-Standard Records,Consistency (%),Status,Recommended Mapping
0,Protocol used for PCF,866,865,1,99.88,WARNING,Traci 2.1 -> TRACI 2.1
1,Country (where company is incorporated),866,866,0,100.00,PASS,-



Non-standard values:


,Feature,Observed Value,Recommended Value
0,Protocol used for PCF,Traci 2.1,TRACI 2.1


In [16]:
# R05 — Measurement Unit Consistency | Consistency

EXPECTED_UNITS = {
    "Product weight (kg)": "kg",
    "Product's carbon footprint (PCF, kg CO2e)": "kg CO2e",
    "*Carbon intensity": "kg CO2e/kg",
    "*Upstream CO2e (fraction of total PCF)": "%",
    "*Operations CO2e (fraction of total PCF)": "%",
    "*Downstream CO2e (fraction of total PCF)": "%",
    "*Transport CO2e (fraction of total PCF)": "%",
    "*EndOfLife CO2e (fraction of total PCF)": "%"
}


def rule_05_measurement_unit_consistency(df):

    results = []

    for feature, expected_unit in EXPECTED_UNITS.items():

        exists = feature in df.columns

        # Check whether the expected unit is represented
        # in the dataset column definition.
        schema_unit_ok = (
            exists
            and (
                expected_unit == "%"
                or expected_unit.lower() in feature.lower()
            )
        )

        if expected_unit == "%" and exists:

            values = df[feature].dropna()

            if len(values) > 0:
                parseable = values.map(
                    percentage_to_float
                ).notna().mean()
            else:
                parseable = np.nan

            unit_ok = (
                schema_unit_ok
                and parseable == 1.0
            )

        else:
            unit_ok = schema_unit_ok
            parseable = np.nan

        results.append({
            "Feature": feature,
            "Expected Unit": expected_unit,
            "Schema Unit Check": (
                "PASS" if schema_unit_ok else "FAIL"
            ),
            "Unit Representation Check": (
                "PASS" if unit_ok else "FAIL"
            ),
            "Consistency (%)": (
                100.0 if unit_ok else 0.0
            ),
            "Status": (
                "PASS" if unit_ok else "FAIL"
            )
        })

    return pd.DataFrame(results)


r05_results = rule_05_measurement_unit_consistency(df)

r05_score = round(
    r05_results["Consistency (%)"].mean(),
    2
)

print("R05 — Measurement Unit Consistency")
print(f"Consistency Score: {r05_score}%")

display(r05_results)

R05 — Measurement Unit Consistency
Consistency Score: 25.0%


,Feature,Expected Unit,Schema Unit Check,Unit Representation Check,Consistency (%),Status
0,Product weight (kg),kg,PASS,PASS,100.0,PASS
1,"Product's carbon footprint (PCF, kg CO2e)",kg CO2e,PASS,PASS,100.0,PASS
2,*Carbon intensity,kg CO2e/kg,FAIL,FAIL,0.0,FAIL
3,*Upstream CO2e (fraction of total PCF),%,PASS,FAIL,0.0,FAIL
4,*Operations CO2e (fraction of total PCF),%,PASS,FAIL,0.0,FAIL
5,*Downstream CO2e (fraction of total PCF),%,PASS,FAIL,0.0,FAIL
6,*Transport CO2e (fraction of total PCF),%,PASS,FAIL,0.0,FAIL
7,*EndOfLife CO2e (fraction of total PCF),%,PASS,FAIL,0.0,FAIL


In [17]:
# R06 — Range & Statistical Plausibility | Plausibility


RANGE_RULES = {
    "Product weight (kg)": {"min": 0},
    "Product's carbon footprint (PCF, kg CO2e)": {"min": 0},
    "*Carbon intensity": {"min": 0}
}

ROBUST_IQR_MULTIPLIER = 3.0


def robust_upper_bound(series, multiplier=ROBUST_IQR_MULTIPLIER):
    values = pd.to_numeric(series, errors="coerce").dropna()

    if len(values) < 4:
        return np.inf

    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1

    if iqr == 0:
        return float(values.max())

    return float(q3 + multiplier * iqr)


def rule_06_range_validation(df):

    results = []
    details = []

    for feature, limits in RANGE_RULES.items():

        numeric = pd.to_numeric(
            df[feature],
            errors="coerce"
        )

        assessed = numeric.notna()
        checked = int(assessed.sum())

        lower = limits["min"]
        upper = robust_upper_bound(numeric)

        valid_mask = (
            assessed
            & numeric.ge(lower)
            & numeric.le(upper)
        )

        valid = int(valid_mask.sum())
        flagged = checked - valid

        score = calculate_score(
            valid,
            checked
        )

        results.append({
            "Feature": feature,
            "Minimum": lower,
            "Robust Upper Bound": (
                round(upper, 6)
                if np.isfinite(upper)
                else "Not estimated"
            ),
            "Checked Numeric Records": checked,
            "Plausible Records": valid,
            "Flagged Records": flagged,
            "Plausibility (%)": score,
            "Status": (
                "PASS"
                if flagged == 0
                else "WARNING"
            )
        })

        # Keep flagged observations as evidence
        if flagged > 0:

            flagged_values = numeric.loc[
                assessed & ~valid_mask
            ]

            detail = pd.DataFrame({
                "Feature": feature,
                "Observed Value": flagged_values
            })

            details.append(detail)

    report_df = pd.DataFrame(results)

    if details:
        details_df = pd.concat(
            details,
            ignore_index=True
        )
    else:
        details_df = pd.DataFrame()

    return report_df, details_df


r06_results, r06_details = rule_06_range_validation(df)

r06_score = round(
    r06_results["Plausibility (%)"]
    .dropna()
    .mean(),
    2
)

print("R06 — Range & Statistical Plausibility")
print(f"Plausibility Score: {r06_score}%")

display(r06_results)

if not r06_details.empty:
    print("\nFlagged observations:")
    display(r06_details)

R06 — Range & Statistical Plausibility
Plausibility Score: 92.96%


,Feature,Minimum,Robust Upper Bound,Checked Numeric Records,Plausible Records,Flagged Records,Plausibility (%),Status
0,Product weight (kg),0,3997.00,866,859,7,99.19,WARNING
1,"Product's carbon footprint (PCF, kg CO2e)",0,6379.00,866,760,106,87.76,WARNING
2,*Carbon intensity,0,100.04,866,796,70,91.92,WARNING



Flagged observations:


,Feature,Observed Value
0,Product weight (kg),17736.00
1,Product weight (kg),12000.00
2,Product weight (kg),361000.00
3,Product weight (kg),400000.00
4,Product weight (kg),600000.00
...,...,...
178,*Carbon intensity,146.00
179,*Carbon intensity,135.22
180,*Carbon intensity,163.98
181,*Carbon intensity,121.05


In [18]:
# R07 — Cross-Field Validation | Plausibility


def rule_07_cross_field_validation(df):

    results = []

    # Check PCF component fractions


    component_columns = [
        "*Upstream CO2e (fraction of total PCF)",
        "*Operations CO2e (fraction of total PCF)",
        "*Downstream CO2e (fraction of total PCF)",
        "*Transport CO2e (fraction of total PCF)",
        "*EndOfLife CO2e (fraction of total PCF)"
    ]

    component_data = df[component_columns].apply(
        lambda col: col.map(percentage_to_float)
    )

    component_sum = component_data.sum(
        axis=1,
        min_count=1
    )

    component_checked = component_sum.notna()

    component_valid = (
        component_checked
        & component_sum.between(99, 101)
    )

    checked = int(component_checked.sum())
    valid = int(component_valid.sum())
    invalid = checked - valid

    results.append({
        "Check": "PCF component fractions sum to approximately 100%",
        "Checked Records": checked,
        "Valid Records": valid,
        "Invalid Records": invalid,
        "Plausibility (%)": calculate_score(
            valid,
            checked
        ),
        "Status": (
            "PASS"
            if invalid == 0
            else "WARNING"
        )
    })


    # Check Upstream estimated percentage

    upstream_estimated = df[
        "*%Upstream estimated from %Operations"
    ].map(percentage_to_float)

    upstream_checked = upstream_estimated.notna()

    upstream_valid = (
        upstream_checked
        & upstream_estimated.between(0, 100)
    )

    checked = int(upstream_checked.sum())
    valid = int(upstream_valid.sum())
    invalid = checked - valid

    results.append({
        "Check": "Upstream estimated percentage is between 0% and 100%",
        "Checked Records": checked,
        "Valid Records": valid,
        "Invalid Records": invalid,
        "Plausibility (%)": calculate_score(
            valid,
            checked
        ),
        "Status": (
            "PASS"
            if invalid == 0
            else "WARNING"
        )
    })

    # Check  Relative PCF change


    change_column = "Relative change in PCF vs previous"

    if change_column in df.columns:

        change_values = df[change_column].map(
            percentage_to_float
        )

        change_checked = change_values.notna()

        # Relative change is allowed to be -ve or +ve
        change_valid = change_checked

        checked = int(change_checked.sum())
        valid = int(change_valid.sum())
        invalid = checked - valid

        results.append({
            "Check": "Relative PCF change is numeric/parseable",
            "Checked Records": checked,
            "Valid Records": valid,
            "Invalid Records": invalid,
            "Plausibility (%)": calculate_score(
                valid,
                checked
            ),
            "Status": (
                "PASS"
                if invalid == 0
                else "WARNING"
            )
        })

    return pd.DataFrame(results)


r07_results = rule_07_cross_field_validation(df)

r07_score = round(
    r07_results["Plausibility (%)"]
    .dropna()
    .mean(),
    2
)

print("R07 — Cross-Field Validation")
print(f"Plausibility Score: {r07_score}%")

display(r07_results)

R07 — Cross-Field Validation
Plausibility Score: 67.82%


,Check,Checked Records,Valid Records,Invalid Records,Plausibility (%),Status
0,PCF component fractions sum to approximately 100%,421,150,271,35.63,WARNING
1,Upstream estimated percentage is between 0% an...,0,0,0,NaN,PASS
2,Relative PCF change is numeric/parseable,250,250,0,100.00,PASS


In [19]:
# R08 — Identifier Uniqueness | Consistency


def rule_08_identifier_uniqueness(df):

    identifier_column = "*PCF-ID"

    identifiers = clean_string_series(
        df[identifier_column]
    )

    valid_ids = identifiers[
        ~is_missing_or_blank(df[identifier_column])
    ]

    total_ids = len(valid_ids)
    unique_ids = valid_ids.nunique()
    duplicate_ids = total_ids - unique_ids

    score = calculate_score(
        unique_ids,
        total_ids
    )

    result = pd.DataFrame([{
        "Feature": identifier_column,
        "Non-Missing IDs": total_ids,
        "Unique IDs": unique_ids,
        "Duplicate IDs": duplicate_ids,
        "Uniqueness (%)": score,
        "Status": (
            "PASS"
            if duplicate_ids == 0
            else "WARNING"
        )
    }])

    duplicate_details = (
        valid_ids[
            valid_ids.duplicated(keep=False)
        ]
        .value_counts()
        .reset_index()
    )

    duplicate_details.columns = [
        "PCF-ID",
        "Occurrences"
    ]

    return result, duplicate_details


r08_results, r08_duplicates = rule_08_identifier_uniqueness(df)

r08_score = round(
    r08_results["Uniqueness (%)"].iloc[0],
    2
)

print("R08 — Identifier Uniqueness")
print(f"Uniqueness Score: {r08_score}%")

display(r08_results)

if not r08_duplicates.empty:
    print("\nDuplicate PCF identifiers:")
    display(r08_duplicates)

R08 — Identifier Uniqueness
Uniqueness Score: 100.0%


,Feature,Non-Missing IDs,Unique IDs,Duplicate IDs,Uniqueness (%),Status
0,*PCF-ID,866,866,0,100.0,PASS


In [20]:
# R09 — Provenance Availability | Traceability

def rule_09_provenance(df):

    provenance_columns = [
        "*Source for product weight",
        "Protocol used for PCF"
    ]

    results = []

    for column in provenance_columns:

        values = df[column]
        assessed = ~is_missing_or_blank(values)

        checked = int(len(values))
        available = int(assessed.sum())
        missing = checked - available

        score = calculate_score(
            available,
            checked
        )

        results.append({
            "Feature": column,
            "Total Records": checked,
            "Provenance Available": available,
            "Provenance Missing": missing,
            "Traceability (%)": score,
            "Status": (
                "PASS"
                if missing == 0
                else "WARNING"
            )
        })

    return pd.DataFrame(results)


r09_results = rule_09_provenance(df)

r09_score = round(
    r09_results["Traceability (%)"].mean(),
    2
)

print("R09 — Provenance Availability")
print(f"Traceability Score: {r09_score}%")

display(r09_results)

R09 — Provenance Availability
Traceability Score: 100.0%


,Feature,Total Records,Provenance Available,Provenance Missing,Traceability (%),Status
0,*Source for product weight,866,866,0,100.0,PASS
1,Protocol used for PCF,866,866,0,100.0,PASS


In [21]:
# R10 — Metadata Availability | Traceability


METADATA_COLUMNS = [
    "Product name (and functional unit)",
    "Product detail",
    "Company",
    "Country (where company is incorporated)",
    "Company's GICS Industry Group",
    "Company's GICS Industry",
    "*Company's sector"
]


def rule_10_metadata(df):

    results = []

    for column in METADATA_COLUMNS:

        values = df[column]
        available_mask = ~is_missing_or_blank(values)

        total_records = len(values)
        available = int(available_mask.sum())
        missing = total_records - available

        score = calculate_score(
            available,
            total_records
        )

        results.append({
            "Feature": column,
            "Total Records": total_records,
            "Metadata Available": available,
            "Metadata Missing": missing,
            "Traceability (%)": score,
            "Status": (
                "PASS"
                if missing == 0
                else "WARNING"
            )
        })

    return pd.DataFrame(results)


r10_results = rule_10_metadata(df)

r10_score = round(
    r10_results["Traceability (%)"].mean(),
    2
)

print("R10 — Metadata Availability")
print(f"Traceability Score: {r10_score}%")

display(r10_results)

R10 — Metadata Availability
Traceability Score: 99.84%


,Feature,Total Records,Metadata Available,Metadata Missing,Traceability (%),Status
0,Product name (and functional unit),866,866,0,100.00,PASS
1,Product detail,866,856,10,98.85,WARNING
2,Company,866,866,0,100.00,PASS
3,Country (where company is incorporated),866,866,0,100.00,PASS
4,Company's GICS Industry Group,866,866,0,100.00,PASS
5,Company's GICS Industry,866,866,0,100.00,PASS
6,*Company's sector,866,866,0,100.00,PASS


In [22]:
# R11 — Reporting Period Validation | Timeliness

REPORTING_YEAR_MIN = 2015
REPORTING_YEAR_MAX = 2017


def rule_11_reporting_period(df):

    years = pd.to_numeric(
        df["Year of reporting"],
        errors="coerce"
    )

    checked_mask = years.notna()

    valid_mask = (
        checked_mask
        & years.between(
            REPORTING_YEAR_MIN,
            REPORTING_YEAR_MAX
        )
    )

    checked = int(checked_mask.sum())
    valid = int(valid_mask.sum())
    invalid = checked - valid

    score = calculate_score(
        valid,
        checked
    )

    result = pd.DataFrame([{
        "Feature": "Year of reporting",
        "Expected Range": (
            f"{REPORTING_YEAR_MIN}-{REPORTING_YEAR_MAX}"
        ),
        "Checked Records": checked,
        "Valid Records": valid,
        "Invalid Records": invalid,
        "Timeliness (%)": score,
        "Status": (
            "PASS"
            if invalid == 0
            else "WARNING"
        )
    }])

    return result


r11_results = rule_11_reporting_period(df)

r11_score = round(
    r11_results["Timeliness (%)"].iloc[0],
    2
)

print("R11 — Reporting Period Validation")
print(f"Timeliness Score: {r11_score}%")

display(r11_results)

R11 — Reporting Period Validation
Timeliness Score: 57.39%


,Feature,Expected Range,Checked Records,Valid Records,Invalid Records,Timeliness (%),Status
0,Year of reporting,2015-2017,866,497,369,57.39,WARNING


In [23]:
### R12 — Freshness Assessment

# R12 was not implemented because the Carbon Catalogue is a static historical dataset and does not provide sufficient information to assess data freshness reliably.
#Therefore, R12 is excluded from the BEACON scoring calculation.

In [24]:
# R13 — Documentation Check | Interpretability

DOCUMENTATION_COLUMN = "*Adjustments to raw data (if any)"


def rule_13_documentation(df):

    values = df[DOCUMENTATION_COLUMN]

    documented_mask = ~is_missing_or_blank(values)

    total_records = len(values)
    documented = int(documented_mask.sum())
    missing = total_records - documented

    score = calculate_score(
        documented,
        total_records
    )

    result = pd.DataFrame([{
        "Feature": DOCUMENTATION_COLUMN,
        "Total Records": total_records,
        "Documented Records": documented,
        "Missing Documentation": missing,
        "Interpretability (%)": score,
        "Status": (
            "PASS"
            if missing == 0
            else "WARNING"
        )
    }])

    return result


r13_results = rule_13_documentation(df)

r13_score = round(
    r13_results["Interpretability (%)"].iloc[0],
    2
)

print("R13 — Documentation Check")
print(f"Interpretability Score: {r13_score}%")

display(r13_results)

R13 — Documentation Check
Interpretability Score: 21.59%


,Feature,Total Records,Documented Records,Missing Documentation,Interpretability (%),Status
0,*Adjustments to raw data (if any),866,187,679,21.59,WARNING


In [25]:
# R14 — Category Coverage | Representativeness

CATEGORY_COLUMNS = [
    "Country (where company is incorporated)",
    "Company's GICS Industry Group",
    "Company's GICS Industry",
    "*Company's sector"
]

MIN_CATEGORY_COUNT = 1


def rule_14_category_coverage(df):

    results = []

    for column in CATEGORY_COLUMNS:

        values = clean_string_series(df[column])

        valid_values = values[
            ~is_missing_or_blank(df[column])
        ]

        category_counts = valid_values.value_counts()

        total_categories = len(category_counts)

        covered_categories = int(
            (category_counts >= MIN_CATEGORY_COUNT).sum()
        )

        score = calculate_score(
            covered_categories,
            total_categories
        )

        results.append({
            "Feature": column,
            "Observed Categories": total_categories,
            "Covered Categories": covered_categories,
            "Coverage (%)": score,
            "Status": (
                "PASS"
                if covered_categories == total_categories
                else "WARNING"
            )
        })

    return pd.DataFrame(results)


r14_results = rule_14_category_coverage(df)

r14_score = round(
    r14_results["Coverage (%)"].mean(),
    2
)

print("R14 — Category Coverage")
print(f"Representativeness Score: {r14_score}%")

display(r14_results)

R14 — Category Coverage
Representativeness Score: 100.0%


,Feature,Observed Categories,Covered Categories,Coverage (%),Status
0,Country (where company is incorporated),28,28,100.0,PASS
1,Company's GICS Industry Group,30,30,100.0,PASS
2,Company's GICS Industry,35,35,100.0,PASS
3,*Company's sector,8,8,100.0,PASS


In [26]:
# R15 — Distribution Balance Assessment | Representativeness


DISTRIBUTION_COLUMNS = [
    "Country (where company is incorporated)",
    "Company's GICS Industry Group",
    "Company's GICS Industry",
    "*Company's sector"
]


def rule_15_distribution_balance(df):

    results = []

    for column in DISTRIBUTION_COLUMNS:

        values = clean_string_series(df[column])

        valid_values = values[
            ~is_missing_or_blank(df[column])
        ]

        counts = valid_values.value_counts()

        total = counts.sum()

        if total == 0:
            score = np.nan
            largest_share = np.nan
        else:
            largest_share = (counts.max() / total) * 100

            # Distribution score based on the largest
            # category share.
            score = round(
                100 - largest_share,
                2
            )

        results.append({
            "Feature": column,
            "Number of Categories": len(counts),
            "Largest Category Share (%)": (
                round(largest_share, 2)
                if not pd.isna(largest_share)
                else np.nan
            ),
            "Distribution Score (%)": score
        })

    return pd.DataFrame(results)


r15_results = rule_15_distribution_balance(df)

r15_score = round(
    r15_results["Distribution Score (%)"].dropna().mean(),
    2
)

print("R15 — Distribution Balance Assessment")
print(f"Representativeness Score: {r15_score}%")

display(r15_results)

R15 — Distribution Balance Assessment
Representativeness Score: 72.06%


,Feature,Number of Categories,Largest Category Share (%),Distribution Score (%)
0,Country (where company is incorporated),28,35.22,64.78
1,Company's GICS Industry Group,30,22.52,77.48
2,Company's GICS Industry,35,24.83,75.17
3,*Company's sector,8,29.21,70.79


In [27]:
# BEACON Rule Results


rule_results = [
    ["R01", "Missing Value Check", "Completeness", r01_score],
    ["R02", "Data Type Validation", "Validity", r02_score],
    ["R03", "Domain Validation", "Validity", r03_score],
    ["R04", "Controlled Vocabulary Check", "Consistency", r04_score],
    ["R05", "Measurement Unit Consistency", "Consistency", r05_score],
    ["R06", "Range & Statistical Plausibility", "Plausibility", r06_score],
    ["R07", "Cross-Field Validation", "Plausibility", r07_score],
    ["R08", "Identifier Uniqueness Check", "Consistency", r08_score],
    ["R09", "Provenance Availability Check", "Traceability", r09_score],
    ["R10", "Metadata Availability Check", "Traceability", r10_score],
    ["R11", "Reporting Period Validation", "Timeliness", r11_score],
    ["R13", "Documentation Check", "Interpretability", r13_score],
    ["R14", "Category Coverage Check", "Representativeness", r14_score],
    ["R15", "Distribution Balance Assessment", "Representativeness", r15_score]
]

rule_matrix_df = pd.DataFrame(
    rule_results,
    columns=[
        "Rule ID",
        "Rule Name",
        "Dimension",
        "Score (%)"
    ]
)

display(rule_matrix_df)

,Rule ID,Rule Name,Dimension,Score (%)
0,R01,Missing Value Check,Completeness,99.95
1,R02,Data Type Validation,Validity,100.00
2,R03,Domain Validation,Validity,98.54
3,R04,Controlled Vocabulary Check,Consistency,99.94
4,R05,Measurement Unit Consistency,Consistency,25.00
5,R06,Range & Statistical Plausibility,Plausibility,92.96
6,R07,Cross-Field Validation,Plausibility,67.82
7,R08,Identifier Uniqueness Check,Consistency,100.00
8,R09,Provenance Availability Check,Traceability,100.00
9,R10,Metadata Availability Check,Traceability,99.84


In [28]:
# Dimension Scores
dimension_scores = (
    rule_matrix_df
    .dropna(subset=["Score (%)"])
    .groupby("Dimension", as_index=False)["Score (%)"]
    .mean()
)

dimension_scores["Score (%)"] = dimension_scores["Score (%)"].round(2)

display(dimension_scores)

,Dimension,Score (%)
0,Completeness,99.95
1,Consistency,74.98
2,Interpretability,21.59
3,Plausibility,80.39
4,Representativeness,86.03
5,Timeliness,57.39
6,Traceability,99.92
7,Validity,99.27


In [29]:
# Overall BEACON Quality Score

overall_score = round(
    dimension_scores["Score (%)"].mean(),
    2
)

print("=" * 70)
print(f"Overall BEACON Quality Score: {overall_score}%")
print("=" * 70)

Overall BEACON Quality Score: 77.44%
